# 01 — Veri Kontrolü (F3 ADIM 0)

Gerçek Binance verisi indirildikten sonraki bütünlük kontrolü. Kontrol edilenler:

1. Bar sayıları ve tarih aralıkları
2. Boşluklar (gerçek piyasa olayı mı, indirme hatası mı?)
3. **Delist edilmiş paritelerin varlığı** (K-13 — survivorship bias)
4. Yarım mum diske yazılmamış mı (K-05 / K-15)
5. **K-13 sağlık kontrolü**: likidite filtresinden sonra yıl bazında pencere sayısı

> Bu defterin yakaladığı en önemli hata: Binance arşivi 2025-01'den itibaren
> zaman damgalarını mikrosaniye yazıyor. Milisaniye varsayan okuyucu bu barları
> "kapanmamış mum" sanıp **sessizce** düşürüyordu (19 aylık veri kaybı).

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
OUT = Path("notebooks/output"); OUT.mkdir(parents=True, exist_ok=True)
TF = "1d"

from engine.config import TIMEFRAME_MS, LIQUIDITY_TOP_PCT
from engine.ingest.store import read_klines, available_symbols

semboller = available_symbols(TF)
satirlar = []
for s in semboller:
    df = read_klines(s, TF)
    ot = df.open_time.to_numpy()
    gaps = np.diff(ot)
    satirlar.append(dict(
        sembol=s, bar=len(df),
        ilk=pd.to_datetime(ot.min(), unit="ms", utc=True).date(),
        son=pd.to_datetime(ot.max(), unit="ms", utc=True).date(),
        bosluk=int((gaps != TIMEFRAME_MS[TF]).sum()),
        max_bosluk_gun=int(gaps.max() // 86_400_000) if len(gaps) else 0,
    ))
veri = pd.DataFrame(satirlar).sort_values("bar", ascending=False)
print(f"{len(veri)} sembol, toplam {veri.bar.sum():,} bar")
veri.head(10)

1165 sembol, toplam 1,505,471 bar


,sembol,bar,ilk,son,bosluk,max_bosluk_gun
593,LTCBTC,3269,2017-08-17,2026-07-29,0,1
372,ETHBTC,3269,2017-08-17,2026-07-29,0,1
216,BTCUSDT,3269,2017-08-17,2026-07-29,0,1
190,BNBBTC,3269,2017-08-17,2026-07-29,0,1
378,ETHUSDT,3269,2017-08-17,2026-07-29,0,1
573,LINKBTC,3227,2017-09-28,2026-07-29,0,1
555,KNCBTC,3217,2017-09-26,2026-07-17,0,1
369,ETCBTC,3213,2017-10-12,2026-07-29,0,1
294,DASHBTC,3205,2017-10-20,2026-07-29,0,1
1010,TRXBTC,3194,2017-10-31,2026-07-29,0,1


In [2]:
# Boşluklar — hepsi gerçek piyasa olayı mı?
bosluklu = veri[veri.bosluk > 0]
print(f"boşluklu sembol: {len(bosluklu)}/{len(veri)}")
bosluklu[["sembol", "bar", "bosluk", "max_bosluk_gun", "ilk", "son"]]

boşluklu sembol: 43/1165


,sembol,bar,bosluk,max_bosluk_gun,ilk,son
589,LRCBTC,3040,3,35,2017-09-01,2026-02-13
1020,TUSDUSDT,2817,1,166,2018-05-31,2026-07-29
1036,USDCUSDT,2619,1,166,2018-12-15,2026-07-29
764,PHBBTC,2441,1,4,2019-05-24,2026-01-30
284,CVCUSDT,2376,1,154,2019-08-27,2026-07-29
909,SNMBTC,2153,1,4,2017-09-27,2023-08-22
601,LUNAUSDT,2152,1,18,2020-08-21,2026-07-29
952,SUNUSDT,2135,1,4,2020-09-21,2026-07-29
422,FTTUSDT,2104,1,311,2019-12-20,2026-07-29
943,STRAXUSDT,2073,1,8,2020-11-18,2026-07-29


In [3]:
# K-13: delist edilmiş pariteler veri setinde mi?
delist_ornek = ["USTUSDT", "BUSDUSDT", "MATICUSDT", "FTMUSDT", "SRMUSDT"]
for s in delist_ornek:
    if s in semboller:
        r = veri[veri.sembol == s].iloc[0]
        print(f"{s:11s} {r.bar:5d} bar  {r.ilk} -> {r.son}   (son bar = delist tarihi)")
    else:
        print(f"{s:11s} veri setinde yok")

USTUSDT       141 bar  2021-12-24 -> 2022-05-13   (son bar = delist tarihi)
BUSDUSDT     1548 bar  2019-09-20 -> 2023-12-15   (son bar = delist tarihi)
MATICUSDT    1965 bar  2019-04-26 -> 2024-09-10   (son bar = delist tarihi)
FTMUSDT      2044 bar  2019-06-11 -> 2025-01-13   (son bar = delist tarihi)
SRMUSDT       840 bar  2020-08-11 -> 2022-11-28   (son bar = delist tarihi)


In [4]:
# K-05 / K-15: yarım mum diske yazılmamalı
import time
simdi = int(time.time() * 1000)
son_bar = int(max(read_klines(s, TF, columns=["open_time"]).open_time.max() for s in semboller))
kapanis = son_bar + TIMEFRAME_MS[TF]
print("en son bar açılışı :", pd.to_datetime(son_bar, unit="ms", utc=True))
print("o barın kapanışı   :", pd.to_datetime(kapanis, unit="ms", utc=True))
print("şu an              :", pd.to_datetime(simdi, unit="ms", utc=True))
print("YARIM MUM YOK" if kapanis <= simdi else "HATA: kapanmamış mum diske yazılmış!")

en son bar açılışı : 2026-07-29 00:00:00+00:00
o barın kapanışı   : 2026-07-30 00:00:00+00:00
şu an              : 2026-07-30 21:09:01.705000+00:00
YARIM MUM YOK


In [5]:
# K-13 SAĞLIK KONTROLÜ — likidite filtresinden sonra yıl bazında pencere
from engine.config import WINDOW_LENGTHS
from engine.matrix.build import matrix_dir, META_FILE
from engine.matrix.npy_store import load_matrix

parcalar = []
for L in WINDOW_LENGTHS:
    M = np.asarray(load_matrix(matrix_dir(TF, L) / META_FILE))
    parcalar.append(pd.DataFrame({
        "yil": pd.to_datetime(M["end_time"], unit="ms", utc=True).year,
        "likit": M["liquid"],
    }))
pencere = pd.concat(parcalar)
saglik = pencere.groupby("yil").agg(toplam=("likit", "size"), likit=("likit", "sum"))
saglik["oran_%"] = (100 * saglik.likit / saglik.toplam).round(1)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(saglik.index, saglik.likit, color="#2b7a78")
ax.set_title(f"Likidite filtresinden geçen pencere (hedef ~%{LIQUIDITY_TOP_PCT*100:.0f} kesitsel)")
ax.set_xlabel("yıl"); ax.set_ylabel("pencere")
fig.tight_layout(); fig.savefig(OUT / "k13_yil_bazinda_pencere.png", dpi=110)
saglik

,toplam,likit,oran_%
yil,,,
2017,3071,1734,56.5
2018,63232,24248,38.3
2019,93563,33881,36.2
2020,146055,55589,38.1
2021,251427,100875,40.1
2022,306755,122400,39.9
2023,302989,120606,39.8
2024,313695,124540,39.7
2025,296869,114789,38.7


**Beklenen:** 2017–2019 çökmemeli. Çökmüşse aylık kesitsel filtre yanlış
uygulanmış, sabit eşik gibi davranıyor demektir (K-13.2).